# UEBA: User Behaviour Analytics

## What This Is
User and Entity Behaviour Analytics (UEBA) detects insider threats and compromised accounts by building behavioral baselines and detecting deviations. Key principle: legitimate users behave consistently; compromised accounts or malicious insiders deviate from their baseline.

## Baseline Modeling
For each user, build a statistical model of their normal behavior:
- Login times (hour of day distribution)
- Login locations (device/IP clustering)
- Data access patterns (which systems, how much data)
- Authentication failures over time
- After-hours activity

## Detection Approaches
- **LSTM autoencoder**: Encode 7-day user activity sequence, detect high reconstruction error
- **Isolation Forest**: Unsupervised anomaly detection on behavioral feature vectors
- **Peer group analysis**: Compare user to role-similar peers; flag outliers within peer group

In [ ]:
import numpy as np
from typing import Dict, List, Tuple

np.random.seed(42)

# Simulate authentication log data
# Features per login event: hour, day_of_week, source_ip_hash, 
# auth_type (password=0, mfa=1, sso=2), success, data_mb_accessed

def generate_user_logs(user_id: int, n_days: int = 60, 
                        compromise_at_day: int = None) -> List[Dict]:
    """Generate authentication logs for a user."""
    logs = []
    
    # User baseline profile
    work_hour_mean = np.random.uniform(9, 11)  # typical work start
    work_hour_std = np.random.uniform(0.5, 1.5)
    typical_ips = np.random.randint(100, 200, 3)  # 2-3 regular IPs
    typical_data_mb = np.random.uniform(50, 500)  # daily data access
    
    for day in range(n_days):
        is_weekend = day % 7 in [5, 6]
        is_compromised = compromise_at_day and day >= compromise_at_day
        
        if is_weekend and not is_compromised:
            # Low weekend activity
            if np.random.random() < 0.8:
                continue
        
        n_logins = np.random.poisson(3 if not is_compromised else 12)
        
        for _ in range(n_logins):
            if is_compromised:
                # Attacker behavior: odd hours, new IPs, high data access
                hour = np.random.choice([2, 3, 4, 23, 22])  # late night
                ip = np.random.randint(200, 300)  # new IP range
                auth_type = 0  # password only (bypassed MFA somehow)
                data_mb = np.random.uniform(1000, 5000)  # exfiltration
                fail_count = 0  # successful after brute force
            else:
                hour = max(6, min(22, np.random.normal(work_hour_mean, work_hour_std)))
                ip = np.random.choice(typical_ips)
                auth_type = np.random.choice([0, 1, 2], p=[0.3, 0.5, 0.2])
                data_mb = np.random.exponential(typical_data_mb / 3)
                fail_count = np.random.poisson(0.1)  # occasional typos
            
            logs.append({
                'day': day, 'hour': hour, 'ip': ip,
                'auth_type': auth_type, 'data_mb': data_mb,
                'fail_count': fail_count, 'is_compromised': int(is_compromised)
            })
    
    return logs

# Generate logs for 20 users, 2 compromised
all_logs = {}
for uid in range(18):
    all_logs[uid] = generate_user_logs(uid, n_days=60)
for uid in [18, 19]:
    all_logs[uid] = generate_user_logs(uid, n_days=60, compromise_at_day=45)

total_logs = sum(len(v) for v in all_logs.values())
print(f'Generated {total_logs} login events for 20 users (2 compromised)')


In [ ]:
# Aggregate into daily behavioral feature vectors

def aggregate_daily_features(logs: List[Dict], day: int) -> np.ndarray:
    """Aggregate one day's logs into a behavioral feature vector."""
    day_logs = [l for l in logs if l['day'] == day]
    if not day_logs:
        return np.zeros(8)
    
    hours = [l['hour'] for l in day_logs]
    ips = [l['ip'] for l in day_logs]
    data_mb = [l['data_mb'] for l in day_logs]
    fail_counts = [l['fail_count'] for l in day_logs]
    auth_types = [l['auth_type'] for l in day_logs]
    
    return np.array([
        len(day_logs),                      # login count
        np.mean(hours),                     # avg hour
        np.std(hours) if len(hours) > 1 else 0,  # hour spread
        len(set(ips)),                      # distinct IPs
        np.sum(data_mb),                    # total data accessed MB
        np.sum(fail_counts),                # auth failures
        np.mean([a == 0 for a in auth_types]),  # fraction password-only
        1.0 if any(h < 6 or h > 22 for h in hours) else 0.0  # after-hours
    ])

# Build per-user daily feature matrices
def build_user_features(logs, n_days=60):
    return np.array([aggregate_daily_features(logs, d) for d in range(n_days)])

user_features = {uid: build_user_features(logs) for uid, logs in all_logs.items()}

# UEBA anomaly detection: Mahalanobis distance from personal baseline
def ueba_score(features: np.ndarray, baseline_days: int = 30) -> np.ndarray:
    """Score each day using Mahalanobis distance from personal baseline."""
    baseline = features[:baseline_days]
    mu = baseline.mean(0)
    cov = np.cov(baseline.T) + np.eye(features.shape[1]) * 0.1
    cov_inv = np.linalg.inv(cov)
    
    scores = []
    for day_feat in features:
        diff = day_feat - mu
        scores.append(np.sqrt(diff @ cov_inv @ diff))
    return np.array(scores)

# Evaluate anomaly scores for compromised vs clean users
print('UEBA Anomaly Scores (baseline: days 0-29, eval: days 45-59)')
print('User | Avg Pre-Compromise Score | Avg Post-Compromise Score')
print('-' * 65)

for uid in range(20):
    scores = ueba_score(user_features[uid])
    pre_score = scores[30:45].mean()
    post_score = scores[45:].mean()
    is_comp = uid >= 18
    marker = ' <-- COMPROMISED' if is_comp else ''
    print(f'  {uid:2d}  |  {pre_score:6.2f}                   |  {post_score:6.2f}{marker}')
